# 02 — 全量抽取 URFD(70 支序列 × 2 模型)

**用途**:下載 UR Fall Detection Dataset 全部 70 支序列(30 falls + 40 adls)→ 重組 30fps mp4 →
用 `yolo26n-pose` 與 `yolo26s-pose` 各抽一份 keypoint cache(parquet)存到 Google Drive。
之後的調參(M4)、benchmark(M5)都直接讀 Drive 上的 cache,不必重跑推論。

**執行方式**:`Runtime → Run all`。**這是長工作**(下載 ~4.5GB + 70×2 支影片跑推論),
實際耗時視官方伺服器速度與 GPU 排隊情況而定,可能 20 分鐘到超過 1 小時不等。

**重要**:
- 會跳出 Google Drive 掛載授權視窗,請允許(資料與 cache 都存在
  `Drive/fall-detection-pose/`,VM 斷線/逾時不會遺失進度)。
- 確認 Drive 至少有 ~5GB 可用空間。
- 全程 idempotent + skip-existing:中途斷線只要重新 `Run all`,已完成的下載/重組/抽取
  都會直接跳過,只補做剩下的部分,不會重工。
- 這本 notebook 只需要成功跑完一次;之後的 notebook 都重用 Drive 上的產物。


In [ ]:
!nvidia-smi || echo '無 GPU:CPU 也能跑,但抽取 70x2 支影片會非常慢,不建議'


In [ ]:
import os
if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        !git clone -q https://github.com/tun0000/fall-detection-pose.git
    %cd fall-detection-pose
!pip -q install -e ".[infer]" pytest

# Colab 在同一個 kernel process 裡直接 pip install -e 之後,site.py 只在
# 直譯器啟動時掃描過一次 .pth,不會自動認得剛裝好的 editable 套件路徑,
# 導致緊接著 import 會是 ModuleNotFoundError(經典一次性 install 的坑,
# 通常要重啟 runtime 才會生效——但這樣就沒辦法一鍵 Run all 了)。
# reload(site) 強迫重新處理 site-packages 下的 .pth;
# 再手動把 src/ 加進 sys.path 當雙重保險(不依賴 hatchling 編輯安裝的內部機制)。
import importlib
import site
import sys

importlib.reload(site)
sys.path.insert(0, os.path.abspath("src"))

import fall_detection
print('fall_detection', fall_detection.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fall-detection-pose'
DATA_DIR = f'{DRIVE_ROOT}/data/urfd'
CACHE_DIR = f'{DRIVE_ROOT}/cache'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print('Drive 掛載完成,資料/快取都會落在:', DRIVE_ROOT)


In [ ]:
# 規則引擎單元測試(純 CPU、合成軌跡,~10 秒):必須全綠
!python -m pytest -q


In [ ]:
# 下載兩份標註 CSV(官方站,CC BY-NC-SA 4.0)
from fall_detection.io import urfd

ann_paths = urfd.download_annotations(DATA_DIR)
ann_paths


In [ ]:
# 下載全部 70 支序列的 cam0 RGB zip(約 4.5GB)。
# 官方站是小型大學伺服器:逐檔下載 + 禮貌間隔 + 全程 skip-existing——
# 斷線重跑不會重工,可以放心中斷。
sequences = urfd.all_sequences()
print(f'共 {len(sequences)} 支序列(30 falls + 40 adls)')
urfd.download_sequences(DATA_DIR, sequences)


In [ ]:
# PNG 序列 zip → 30fps mp4(同樣 skip-existing)
videos = urfd.build_videos(DATA_DIR, sequences)
print(f'重組完成:{len(videos)} 支影片')


In [ ]:
# tune/test 名單已在規劃階段以 seed=42 產生並存進版控(eval/splits.yaml),
# 這裡只讀取、不重新產生——確保 test split 不會因執行環境差異而悄悄漂移
from fall_detection.eval.splits import load_splits

splits = load_splits('eval/splits.yaml')
print('seed =', splits['seed'])
print('tune:', len(splits['tune']['falls']), 'falls +', len(splits['tune']['adls']), 'adls')
print('test:', len(splits['test']['falls']), 'falls +', len(splits['test']['adls']), 'adls')


In [ ]:
# 抽取 keypoint cache:yolo26n-pose(唯一 GPU 步驟;每支影片各自 reset tracker)。
# 全程 skip-existing:斷線重跑只會補完尚未抽取的影片。
import torch
from fall_detection.config import load_config
from fall_detection.inference.extract import extract_batch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
cfg_n = load_config('config.yaml')
cfg_n.model.name = 'yolo26n-pose.pt'
print('model =', cfg_n.model.name, '| device =', device)

video_paths = [videos[seq] for seq in sequences]
extract_batch(video_paths, f'{CACHE_DIR}/yolo26n-pose', cfg_n, device=device)


In [ ]:
# 抽取 keypoint cache:yolo26s-pose(模型較大、精度較高,單獨一格方便追蹤進度/續跑)
cfg_s = load_config('config.yaml')
cfg_s.model.name = 'yolo26s-pose.pt'
print('model =', cfg_s.model.name, '| device =', device)

extract_batch(video_paths, f'{CACHE_DIR}/yolo26s-pose', cfg_s, device=device)


In [ ]:
# 驗證:cache 覆蓋 70 支 × 2 模型、meta 完整;GT 事件數 = 30(每支 fall 恰一次)
from pathlib import Path
from fall_detection.io.cache import read_cache
from fall_detection.eval.ground_truth import load_gt_events

missing = []
cache_rows = {'yolo26n-pose': 0, 'yolo26s-pose': 0}
for model_dir in ('yolo26n-pose', 'yolo26s-pose'):
    for seq in sequences:
        p = Path(f'{CACHE_DIR}/{model_dir}/{seq}.parquet')
        if not p.exists():
            missing.append(str(p))
            continue
        df, meta = read_cache(p)
        assert meta.model_name.startswith(model_dir), f'{p}: model_name={meta.model_name}'
        cache_rows[model_dir] += len(df)

gt_falls = load_gt_events(f'{DATA_DIR}/urfall-cam0-falls.csv')
gt_adls = load_gt_events(f'{DATA_DIR}/urfall-cam0-adls.csv')

print('missing cache files:', missing if missing else '無(70/70 齊全 x 2 模型)')
print('cache rows:', cache_rows)
print('GT fall 事件數(應為 30):', len(gt_falls))
print('GT adl 誤標事件數(應為 0):', len(gt_adls))


In [ ]:
import json

report = {
    'n_sequences': len(sequences),
    'n_falls': len(urfd.fall_sequences()),
    'n_adls': len(urfd.adl_sequences()),
    'missing_cache_files': missing,
    'cache_rows': cache_rows,
    'gt_fall_events': len(gt_falls),
    'gt_adl_events': len(gt_adls),
    'splits_seed': splits['seed'],
    'splits_counts': {
        'tune_falls': len(splits['tune']['falls']),
        'tune_adls': len(splits['tune']['adls']),
        'test_falls': len(splits['test']['falls']),
        'test_adls': len(splits['test']['adls']),
    },
}
print('=== M3 摘要(請貼回對話) ===')
print(json.dumps(report, ensure_ascii=False, indent=2))
